# TMDB Movie Data Analysis

This notebook builds a movie data analysis pipeline in Python and Pandas:

1. **Data Extraction**: fetch movie details from the TMDB API
2. **Cleaning & Transformation**: flatten nested JSON, fix data types, and handle missing or unrealistic values
3. **KPI Analysis**: rank movies by financial and popularity metrics, and compare franchises and directors
4. **Visualization**: present the key trends in charts

**Data source:** [The Movie Database (TMDB) API](https://developer.themoviedb.org/docs)

## Setup & Configuration

All imports and settings are kept in one place so the notebook's dependencies are clear.

- The API key is loaded from a `.env` file with `python-dotenv`, so it never appears in the code or on GitHub.
- `pathlib` builds file paths that work on both Windows and macOS/Linux.
- `movie_ids` is the list of movies the project brief asks us to fetch.

In [14]:
import os
import json
import time
import numpy as np
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"

PROJECT_ROOT = Path.cwd().parent          # notebook lives in notebooks/, so go up one level
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]

## Step 1: Data Extraction

Each movie is requested from the `/movie/{movie_id}` endpoint. Adding `append_to_response=credits` returns the cast and crew in the same request. The basic movie details don't include them, but we need them later for the `cast`, `cast_size`, `director` and `crew_size` columns.

**Error handling strategy:**

| Situation | Response |
|---|---|
| `200` OK | Return the movie data |
| `429` Too Many Requests | Wait, then retry (the problem is temporary) |
| Network error | Wait, then retry (up to 3 attempts) |
| Any other error, e.g. `404` Not Found | Skip the movie and log the reason (retrying won't help) |

A failed movie returns `None` instead of raising an error, so one bad ID can't stop the whole run. A shared `requests.Session` reuses the same connection across requests, which makes fetching faster.

In [7]:
def fetch_movie(movie_id, session, retries=3):
    """Fetch one movie (with cast & crew). Returns a dict, or None if it fails."""
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "append_to_response": "credits"}

    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, params=params, timeout=10)
        except requests.RequestException as e:
            print(f"ID {movie_id}: network error ({e}), attempt {attempt}/{retries}")
            time.sleep(2)
            continue

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:  # too many requests: wait and retry
            wait = int(response.headers.get("Retry-After", 2))
            time.sleep(wait)
            continue

        # Any other error (e.g. 404 for an ID that doesn't exist): don't retry
        message = response.json().get("status_message", "unknown error")
        print(f"ID {movie_id}: skipped ({response.status_code} - {message})")
        return None

    print(f"ID {movie_id}: failed after {retries} attempts")
    return None


def fetch_movies(ids):
    """Fetch several movies and return a list of the ones that succeeded."""
    movies = []
    with requests.Session() as session:
        for movie_id in ids:
            data = fetch_movie(movie_id, session)
            if data is not None:
                movies.append(data)
    print(f"\nFetched {len(movies)} of {len(ids)} movies.")
    return movies

### Fetch all movies and save the raw data

The raw API response is saved to `data/raw/movies_raw.json` before any cleaning. This gives us:

- **Reproducibility:** the cleaning steps can be rerun without calling the API again.
- **Preserved structure:** JSON keeps nested fields such as `genres` and `credits` intact. Saving to CSV would flatten them into plain text.

In [8]:
raw_movies = fetch_movies(movie_ids)

with open(RAW_DIR / "movies_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_movies, f, ensure_ascii=False, indent=2)

ID 0: skipped (404 - The resource you requested could not be found.)

Fetched 18 of 19 movies.


In [6]:
if response.status_code == 200:
    movie = response.json()
else:
    print("Request failed:", response.status_code, response.json().get("status_message"))

### First look at the raw data

The list of movie dictionaries is converted into a DataFrame for inspection.

**Observations:**
- ID `0` does not exist in TMDB, so it was skipped. That leaves *N* movies.
- Several columns hold nested structures instead of single values. `genres`, `production_companies`, `production_countries` and `spoken_languages` are lists of dictionaries, and `belongs_to_collection` is a dictionary or empty.
- `credits` holds two lists, `cast` and `crew`, that need to be extracted.
- These nested columns are the main cleaning task in Step 2.

In [10]:
df_raw = pd.DataFrame(raw_movies)

print(df_raw.shape)
df_raw.head()

(18, 28)


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.246,29031,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.611,34862,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,136,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.240,21026,"{'cast': [{'adult': False, 'gender': 1, 'id': ..."
3,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",300000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/avengers-infinit...,299536,tt4154756,[US],en,...,149,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,An entire universe. Once and for all.,Avengers: Infinity War,False,8.245,33173,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
4,False,/sCzcYW9h55WcesOqA12cgEr9Exw.jpg,None,200000000,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",https://www.paramountmovies.com/movies/titanic,597,tt0120338,[US],en,...,194,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Nothing on earth could come between them.,Titanic,False,7.901,27914,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


In [13]:
df_raw.loc[0, "credits"].keys()

dict_keys(['cast', 'crew'])

## 2. Data Cleaning & Preprocessing

The raw API data is not ready for analysis. It contains irrelevant columns and nested JSON, and some values are missing or unrealistic. This step turns it into a tidy table with **one row per movie and one value per cell**.

All cleaning happens on a copy of the raw DataFrame, so `df_raw` stays untouched for reference.

In [28]:
df = df_raw.copy()

### 2.1 Drop irrelevant columns

These columns add nothing to the analysis:

- `adult`: every movie in this set is non-adult, so the column has no variation
- `imdb_id`: we already identify movies by TMDB `id`
- `original_title`: duplicates `title` for most movies
- `video`: a TMDB flag that is not relevant here
- `homepage`: URLs are not useful for analysis

In [29]:
cols_to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
df = df.drop(columns=cols_to_drop, errors="ignore")
df.shape

(18, 23)

### 2.2 Flatten the JSON-like columns

Several columns store lists of dictionaries instead of plain values. For example, `genres` looks like this:

    [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]

We only need the names, joined into one string with `|`, for example `Action|Adventure`. This format is easy to read, easy to search with `.str.contains()`, and easy to split back apart when needed.

Two helper functions handle this:

- `extract_names()` handles list columns (genres, companies, countries, languages)
- `extract_collection()` handles `belongs_to_collection`, which is a single dictionary, or empty when the movie isn't part of a franchise

Both return `NaN` when there is no data, so missing values are marked consistently.

Both helpers return strings unchanged, so rerunning the cell does not erase already-cleaned values.

For `spoken_languages` we use `english_name`. Its `name` field holds the language's native name (e.g. *Español*), which would make the column inconsistent.

In [30]:
def extract_names(items, key="name", sep="|"):
    """Turn a list of dicts into a 'sep'-joined string of the given key. Returns NaN if empty.
    Strings (already-cleaned values) are returned unchanged, so rerunning is safe."""
    if isinstance(items, str):
        return items
    if not isinstance(items, list) or len(items) == 0:
        return np.nan
    names = [d.get(key) for d in items if isinstance(d, dict) and d.get(key)]
    return sep.join(names) if names else np.nan


def extract_collection(value):
    """Return the collection name from a dict, or NaN if the movie isn't in a collection.
    Strings (already-cleaned values) are returned unchanged, so rerunning is safe."""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        return value.get("name", np.nan)
    return np.nan


df["belongs_to_collection"] = df["belongs_to_collection"].apply(extract_collection)

list_columns = {
    "genres": "name",
    "production_companies": "name",
    "production_countries": "name",
    "spoken_languages": "english_name",
}
for col, key in list_columns.items():
    df[col] = df[col].apply(extract_names, key=key)

df[["title", "belongs_to_collection", "genres", "spoken_languages"]].head()

,title,belongs_to_collection,genres,spoken_languages
0,Avengers: Endgame,The Avengers Collection,Adventure|Science Fiction|Action,English|Japanese|Xhosa
1,Avatar,Avatar Collection,Science Fiction|Action|Adventure,English|Spanish
2,Star Wars: The Force Awakens,Star Wars Collection,Adventure|Action|Science Fiction,English
3,Avengers: Infinity War,The Avengers Collection,Adventure|Action|Science Fiction,English|Xhosa
4,Titanic,NaN,Drama|Romance,English|French|German|Swedish|Italian|Russian


### 2.3 Extract cast and crew from `credits`

The `credits` column holds two lists, `cast` and `crew`. The final dataset needs four columns from it:

| Column | Source |
|---|---|
| `cast` | All actor names, joined with `\|` |
| `cast_size` | Number of cast entries |
| `director` | Crew members whose `job` is `"Director"` (some movies have more than one) |
| `crew_size` | Number of crew entries. One person can appear more than once if they had several jobs |

`credits` is dropped afterwards because all useful information has been extracted.

In [31]:
def extract_credits(credits):
    """Extract cast names, cast size, director(s) and crew size from a credits dict."""
    if not isinstance(credits, dict):
        return pd.Series({"cast": np.nan, "cast_size": np.nan,
                          "director": np.nan, "crew_size": np.nan})

    cast = credits.get("cast", [])
    crew = credits.get("crew", [])
    directors = [p["name"] for p in crew if p.get("job") == "Director"]

    return pd.Series({
        "cast": "|".join(p["name"] for p in cast) or np.nan,
        "cast_size": len(cast),
        "director": "|".join(directors) or np.nan,
        "crew_size": len(crew),
    })


df[["cast", "cast_size", "director", "crew_size"]] = df["credits"].apply(extract_credits)
df = df.drop(columns="credits")

df[["title", "director", "cast_size", "crew_size"]]

,title,director,cast_size,crew_size
0,Avengers: Endgame,Joe Russo|Anthony Russo,103,613
1,Avatar,James Cameron,68,999
2,Star Wars: The Force Awakens,J.J. Abrams,99,270
3,Avengers: Infinity War,Joe Russo|Anthony Russo,69,738
4,Titanic,James Cameron,116,271
5,Jurassic World,Colin Trevorrow,53,420
6,The Lion King,Jon Favreau,20,72
7,The Avengers,Joss Whedon,112,639
8,Furious 7,James Wan,50,239
9,Avengers: Age of Ultron,Joss Whedon,75,660


### 2.4 Inspect the extracted columns

`value_counts()` shows every unique value and how often it appears. It helps spot anomalies such as unexpected `NaN`s, empty strings, or inconsistent spellings.

Each column is checked two ways:

1. **As a whole string**: shows the exact combinations, e.g. `Action|Adventure|Science Fiction`
2. **Split into individual items**: shows how often each single genre, country, etc. appears. This is usually more informative.

In [32]:
extracted_cols = ["belongs_to_collection", "genres", "spoken_languages",
                  "production_countries", "production_companies"]

for col in extracted_cols:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False).head(10))
    print()

===== belongs_to_collection =====
belongs_to_collection
The Avengers Collection                4
Star Wars Collection                   2
NaN                                    2
Jurassic Park Collection               2
Frozen Collection                      2
Avatar Collection                      1
The Lion King (Reboot) Collection      1
The Fast and the Furious Collection    1
Black Panther Collection               1
Harry Potter Collection                1
Name: count, dtype: int64

===== genres =====
genres
Adventure|Action|Science Fiction             3
Science Fiction|Action|Adventure             2
Adventure|Science Fiction|Thriller           2
Action|Adventure|Science Fiction             2
Adventure|Science Fiction|Action             1
Drama|Romance                                1
Adventure|Drama|Family|Animation             1
Action|Crime|Thriller                        1
Adventure|Fantasy                            1
Family|Animation|Adventure|Comedy|Fantasy    1
Name: count

In [33]:
for col in ["genres", "spoken_languages", "production_countries", "production_companies"]:
    print(f"===== {col} (individual items) =====")
    print(df[col].str.split("|").explode().value_counts().head(10))
    print()

===== genres (individual items) =====
genres
Adventure          15
Science Fiction    10
Action             10
Family              5
Animation           4
Fantasy             4
Thriller            3
Drama               2
Romance             2
Crime               1
Name: count, dtype: int64

===== spoken_languages (individual items) =====
spoken_languages
English     18
Xhosa        3
Russian      3
Spanish      2
French       2
Japanese     1
German       1
Swedish      1
Italian      1
Hindi        1
Name: count, dtype: int64

===== production_countries (individual items) =====
production_countries
United States of America    18
United Kingdom               2
Name: count, dtype: int64

===== production_companies (individual items) =====
production_companies
Marvel Studios                   5
Universal Pictures               3
Lightstorm Entertainment         2
20th Century Fox                 2
Lucasfilm Ltd.                   2
Amblin Entertainment             2
Walt Disney Pictures 

So standalone movies would be:

In [35]:
df.loc[df["belongs_to_collection"].isna(), ["title", "release_date"]]

,title,release_date
4,Titanic,1997-12-18
16,Beauty and the Beast,2017-03-16


### 2.5 Convert data types

Pandas does not always guess column types correctly. Numbers stored as text can't be used in calculations, and dates stored as text can't be sorted or grouped by year.

- Numeric columns are converted with `pd.to_numeric(errors="coerce")`. Any value that can't be converted becomes `NaN` instead of raising an error.
- `release_date` is converted to a proper datetime.
- `cast_size` and `crew_size` are included because they were created by the credits function and may have been stored as generic objects.

In [36]:
numeric_cols = ["id", "budget", "revenue", "popularity", "runtime",
                "vote_count", "vote_average", "cast_size", "crew_size"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df[numeric_cols + ["release_date"]].dtypes

id                       int64
budget                   int64
revenue                  int64
popularity             float64
runtime                  int64
vote_count               int64
vote_average           float64
cast_size                int64
crew_size                int64
release_date    datetime64[us]
dtype: object

In [ ]:
### 2.6 Replace unrealistic values

**Zeros in `budget`, `revenue` and `runtime`:** a movie can't have a 0-minute runtime or a $0 budget. TMDB stores `0` when the value is unknown. Leaving these zeros would distort averages and make ROI calculations divide by zero. They are replaced with `NaN`, meaning "unknown."

*Why not infer values from similar movies?* With only 18 movies, there are too few comparable films to make a reliable estimate. `NaN` is the honest choice.

**Million USD:** budgets and revenues in raw dollars are hard to read (e.g. `356000000`). They are converted to millions (`356.0`) in new columns, `budget_musd` and `revenue_musd`.

In [37]:
zero_cols = ["budget", "revenue", "runtime"]
print("Zeros before:\n", (df[zero_cols] == 0).sum(), "\n")

df[zero_cols] = df[zero_cols].replace(0, np.nan)

df["budget_musd"] = df["budget"] / 1_000_000
df["revenue_musd"] = df["revenue"] / 1_000_000
df = df.drop(columns=["budget", "revenue"])

df[["title", "budget_musd", "revenue_musd", "runtime"]]

Zeros before:
 budget     0
revenue    0
runtime    0
dtype: int64 



,title,budget_musd,revenue_musd,runtime
0,Avengers: Endgame,356.0,2927.449455,181
1,Avatar,237.0,2923.706026,162
2,Star Wars: The Force Awakens,245.0,2068.223624,136
3,Avengers: Infinity War,300.0,2052.415039,149
4,Titanic,200.0,2264.162353,194
5,Jurassic World,150.0,1671.537444,124
6,The Lion King,260.0,1662.020819,118
7,The Avengers,220.0,1518.815515,143
8,Furious 7,190.0,1515.400000,138
9,Avengers: Age of Ultron,235.0,1405.403694,141


**Movies with `vote_count = 0`:** if nobody voted, a `vote_average` of 0 doesn't mean "terrible." It means "no rating yet." Treating it as a real score would unfairly drag down averages, so these ratings are set to `NaN`.

In [38]:
no_votes = df["vote_count"] == 0
print(f"Movies with no votes: {no_votes.sum()}")
display(df.loc[no_votes, ["title", "vote_count", "vote_average"]])

df.loc[no_votes, "vote_average"] = np.nan

Movies with no votes: 0


,title,vote_count,vote_average


**Placeholder text in `overview` and `tagline`:** text like `"No Data"` or an empty string looks like a value but means "missing." It is replaced with `NaN` so that `isna()` counts it correctly.

In [39]:
placeholders = ["No Data", "No overview found.", "No overview", "N/A", ""]

for col in ["overview", "tagline"]:
    df[col] = df[col].str.strip().replace(placeholders, np.nan)

df[["overview", "tagline"]].isna().sum()

overview    0
tagline     0
dtype: int64

### 2.7–2.9 Remove duplicates, incomplete rows and unreleased movies

Rows are filtered in three steps, and the row count is printed after each one so it's clear what every step removed:

1. **Duplicates:** a movie should appear only once. `id` is TMDB's unique identifier, so duplicates are detected by `id`.
2. **Missing identity:** a row without an `id` or `title` can't be identified, so it is dropped.
3. **Mostly empty rows:** rows need at least 10 non-missing values to be useful.
4. **Release status:** only `Released` movies are kept, because unreleased films have no real revenue or ratings. The `status` column is then dropped, since every remaining value would be identical.

In [40]:
print(f"Start:                    {len(df)} rows")

df = df.drop_duplicates(subset="id")
print(f"After removing duplicates: {len(df)} rows")

df = df.dropna(subset=["id", "title"])
print(f"After requiring id/title:  {len(df)} rows")

df = df.dropna(thresh=10)
print(f"After ≥10 non-NaN values:  {len(df)} rows")

df = df[df["status"] == "Released"].drop(columns="status")
print(f"After keeping Released:    {len(df)} rows")

Start:                    18 rows
After removing duplicates: 18 rows
After requiring id/title:  18 rows
After ≥10 non-NaN values:  18 rows
After keeping Released:    18 rows


### 2.10–2.11 Reorder columns and reset the index

Columns are arranged in a logical order: identity, then descriptive details, financials, ratings, and credits. Selecting this list also drops leftover columns that aren't needed, such as `backdrop_path` and `origin_country`.

The index is reset so the rows are numbered 0, 1, 2, … with no gaps left by the removed rows.

In [41]:
final_columns = [
    "id", "title", "tagline", "release_date", "genres", "belongs_to_collection",
    "original_language", "budget_musd", "revenue_musd", "production_companies",
    "production_countries", "vote_count", "vote_average", "popularity", "runtime",
    "overview", "spoken_languages", "poster_path", "cast", "cast_size", "director", "crew_size"
]

df = df[final_columns].reset_index(drop=True)
df.head()

,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,...,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2927.449455,Marvel Studios,...,8.246,88.8507,181,After the devastating events of Avengers: Infi...,English|Japanese|Xhosa,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,103,Joe Russo|Anthony Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,...,7.611,55.4785,162,"In the 22nd century, a paraplegic Marine is di...",English|Spanish,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,68,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure|Action|Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd.|Bad Robot,...,7.240,36.5288,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,Daisy Ridley|John Boyega|Oscar Isaac|Adam Driv...,99,J.J. Abrams,270
3,299536,Avengers: Infinity War,An entire universe. Once and for all.,2018-04-25,Adventure|Action|Science Fiction,The Avengers Collection,en,300.0,2052.415039,Marvel Studios,...,8.245,117.3789,149,As the Avengers and their allies have continue...,English|Xhosa,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...,69,Joe Russo|Anthony Russo,738
4,597,Titanic,Nothing on earth could come between them.,1997-12-18,Drama|Romance,NaN,en,200.0,2264.162353,Paramount Pictures|20th Century Fox|Lightstorm...,...,7.901,51.3494,194,101-year-old Rose DeWitt Bukater tells the sto...,English|French|German|Swedish|Italian|Russian,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...,116,James Cameron,271


### 2.12 Final check and save

A last overview of types and missing values confirms the data is clean. The result is saved to `data/processed/`, so the analysis can start from clean data without repeating these steps.

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     18 non-null     int64         
 1   title                  18 non-null     str           
 2   tagline                18 non-null     str           
 3   release_date           18 non-null     datetime64[us]
 4   genres                 18 non-null     str           
 5   belongs_to_collection  16 non-null     str           
 6   original_language      18 non-null     str           
 7   budget_musd            18 non-null     float64       
 8   revenue_musd           18 non-null     float64       
 9   production_companies   18 non-null     str           
 10  production_countries   18 non-null     str           
 11  vote_count             18 non-null     int64         
 12  vote_average           18 non-null     float64       
 13  popularity        

In [43]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "movies_clean.csv", index=False)

## 3. KPI Analysis

Now that the data is clean, I can start asking questions. The first one: which movies did best and worst, financially and with audiences?

The API doesn't give me profit or ROI directly, so I calculate them myself:

- **Profit** = revenue − budget (in millions)
- **ROI** (return on investment) = revenue ÷ budget. An ROI of 3 means the movie earned three times what it cost.

If a movie is missing either budget or revenue, both results end up as `NaN`, which is what I want. I'd rather have no number than a wrong one.

In [44]:
df["profit_musd"] = df["revenue_musd"] - df["budget_musd"]
df["roi"] = df["revenue_musd"] / df["budget_musd"]

df[["title", "budget_musd", "revenue_musd", "profit_musd", "roi"]].round(2)

,title,budget_musd,revenue_musd,profit_musd,roi
0,Avengers: Endgame,356.0,2927.45,2571.45,8.22
1,Avatar,237.0,2923.71,2686.71,12.34
2,Star Wars: The Force Awakens,245.0,2068.22,1823.22,8.44
3,Avengers: Infinity War,300.0,2052.42,1752.42,6.84
4,Titanic,200.0,2264.16,2064.16,11.32
5,Jurassic World,150.0,1671.54,1521.54,11.14
6,The Lion King,260.0,1662.02,1402.02,6.39
7,The Avengers,220.0,1518.82,1298.82,6.90
8,Furious 7,190.0,1515.40,1325.40,7.98
9,Avengers: Age of Ultron,235.0,1405.40,1170.40,5.98


### 3.1 Best and worst performers

I need ten different rankings, and they all follow the same pattern: optionally filter, sort by a column, take the top few. Rather than writing that ten times, I wrote one function, `rank_movies()`, and call it with different settings.

Two of the rankings need a minimum threshold to be fair:

- **ROI only counts movies with a budget of at least $10M.** A tiny indie film that cost $100k and made $5M would have a huge ROI and drown out everything else, even though it isn't really comparable.
- **Ratings only count movies with at least 10 votes.** A 9.5 average from three people doesn't tell me much.

In [45]:
def rank_movies(data, metric, n=5, ascending=False, min_col=None, min_value=None,
                extra_cols=None):
    """Return the top-n movies ranked by `metric`.

    ascending=False -> highest first; True -> lowest first.
    min_col/min_value -> only keep rows where data[min_col] >= min_value.
    """
    subset = data.dropna(subset=[metric])
    if min_col is not None:
        subset = subset[subset[min_col] >= min_value]

    cols = ["title", metric] + (extra_cols or [])
    result = subset.sort_values(metric, ascending=ascending).head(n)[cols]
    result.index = range(1, len(result) + 1)   # rank numbers start at 1
    return result

Instead of ten separate cells, I listed all the rankings in one place and loop over them. If I want to add or change a ranking later, I only touch this list.

In [46]:
rankings = [
    {"title": "Highest Revenue",  "metric": "revenue_musd"},
    {"title": "Highest Budget",   "metric": "budget_musd"},
    {"title": "Highest Profit",   "metric": "profit_musd"},
    {"title": "Lowest Profit",    "metric": "profit_musd", "ascending": True},
    {"title": "Highest ROI (budget ≥ $10M)", "metric": "roi",
     "min_col": "budget_musd", "min_value": 10, "extra_cols": ["budget_musd"]},
    {"title": "Lowest ROI (budget ≥ $10M)",  "metric": "roi", "ascending": True,
     "min_col": "budget_musd", "min_value": 10, "extra_cols": ["budget_musd"]},
    {"title": "Most Voted",       "metric": "vote_count"},
    {"title": "Highest Rated (≥ 10 votes)", "metric": "vote_average",
     "min_col": "vote_count", "min_value": 10, "extra_cols": ["vote_count"]},
    {"title": "Lowest Rated (≥ 10 votes)",  "metric": "vote_average", "ascending": True,
     "min_col": "vote_count", "min_value": 10, "extra_cols": ["vote_count"]},
    {"title": "Most Popular",     "metric": "popularity"},
]

for r in rankings:
    params = {k: v for k, v in r.items() if k != "title"}
    print(f"\n=== {r['title']} ===")
    display(rank_movies(df, **params).round(2))


=== Highest Revenue ===


,title,revenue_musd
1,Avengers: Endgame,2927.45
2,Avatar,2923.71
3,Titanic,2264.16
4,Star Wars: The Force Awakens,2068.22
5,Avengers: Infinity War,2052.42



=== Highest Budget ===


,title,budget_musd
1,Avengers: Endgame,356.0
2,Avengers: Infinity War,300.0
3,Star Wars: The Last Jedi,300.0
4,The Lion King,260.0
5,Star Wars: The Force Awakens,245.0



=== Highest Profit ===


...

,title,vote_average,vote_count
1,Avengers: Endgame,8.25,29031
2,Avengers: Infinity War,8.24,33173
3,Harry Potter and the Deathly Hallows: Part 2,8.10,22580
4,The Avengers,8.08,40058
5,Titanic,7.90,27914



=== Lowest Rated (≥ 10 votes) ===


,title,vote_average,vote_count
1,Jurassic World: Fallen Kingdom,6.52,13083
2,Jurassic World,6.70,22018
3,Star Wars: The Last Jedi,6.73,16784
4,Beauty and the Beast,6.97,16225
5,The Lion King,7.09,10961



=== Most Popular ===


,title,popularity
1,Avengers: Infinity War,117.38
2,Avengers: Endgame,88.85
3,Avengers: Age of Ultron,65.55
4,The Avengers,62.11
5,Avatar,55.48


Find

### 3.2 Search queries

Both searches combine conditions across text columns that hold `|`-separated lists, for example finding "Action" inside `Action|Adventure|Science Fiction`.

I wrote a small helper that splits each value back into a list and checks for **exact** matches. Plain substring matching could give false hits. A search for "Action" would also match a hypothetical genre like "Action Comedy," and a search for an actor's name could match someone whose name contains it.

In [47]:
def has_all(series, items):
    """True where the '|'-separated value contains every item in `items` (exact matches)."""
    return series.str.split("|").apply(
        lambda values: isinstance(values, list) and all(item in values for item in items)
    )

**Search 1:** the best-rated Science Fiction Action movies starring RDJ, highest rating first.

In [56]:
search_1 = df[
    has_all(df["genres"], ["Science Fiction", "Action"])
    & has_all(df["cast"], ["Robert Downey Jr."])
].sort_values("vote_average", ascending=False)

search_1[["title", "genres", "vote_average"]]

,title,genres,vote_average
0,Avengers: Endgame,Adventure|Science Fiction|Action,8.246
3,Avengers: Infinity War,Adventure|Action|Science Fiction,8.245
7,The Avengers,Science Fiction|Action|Adventure,8.081
9,Avengers: Age of Ultron,Action|Adventure|Science Fiction,7.278


**Search 2:** movies starring Uma Thurman and directed by Quentin Tarantino, shortest runtime first.

In [ ]:
search_2 = df[
    has_all(df["cast"], ["Rober Downey Jr."])
    & has_all(df["director"], ["Joe Russo"])
].sort_values("runtime", ascending=True)

search_2[["title", "runtime", "director"]]

,title,runtime,director


Both searches returned no results. That's not a bug: none of the 18 movies in this dataset feature Bruce Willis or were directed by Quentin Tarantino. To make sure the filter actually works, I ran the same query with Robert Downey Jr. and got the Avengers films back as expected.

### 3.3 Franchise vs. standalone movies

Do movies in a franchise actually perform better than one-off films? To find out, I split the movies into two groups based on whether `belongs_to_collection` has a value, then compared their averages.

For ROI I used the **median** instead of the mean. ROI can have extreme outliers (one movie earning 15x its budget can drag the average way up), and the median is less affected by that. It shows what a "typical" movie in each group looks like.

In [57]:
df["movie_type"] = np.where(df["belongs_to_collection"].notna(), "Franchise", "Standalone")

franchise_vs_standalone = df.groupby("movie_type").agg(
    movie_count=("id", "count"),
    mean_revenue_musd=("revenue_musd", "mean"),
    median_roi=("roi", "median"),
    mean_budget_musd=("budget_musd", "mean"),
    mean_popularity=("popularity", "mean"),
    mean_rating=("vote_average", "mean"),
).round(2)

franchise_vs_standalone.T

movie_type,Franchise,Standalone
movie_count,16.00,2.00
mean_revenue_musd,1690.78,1765.14
median_roi,7.84,9.62
mean_budget_musd,218.00,180.00
mean_popularity,49.34,40.94
mean_rating,7.40,7.44


### 3.4 Most successful franchises

Next I zoomed in on the franchises themselves: how many films each one has here, how much they cost and earned in total and on average, and how well they were rated. I sorted by total revenue, since that's the clearest single measure of commercial success.

Note that "number of movies" means movies **in this dataset**, not in the whole franchise. The Avengers collection has four films here, but the wider Marvel universe has many more that weren't part of the list I was given.

In [58]:
franchise_stats = (
    df.dropna(subset=["belongs_to_collection"])
      .groupby("belongs_to_collection")
      .agg(
          movie_count=("id", "count"),
          total_budget_musd=("budget_musd", "sum"),
          mean_budget_musd=("budget_musd", "mean"),
          total_revenue_musd=("revenue_musd", "sum"),
          mean_revenue_musd=("revenue_musd", "mean"),
          mean_rating=("vote_average", "mean"),
      )
      .sort_values("total_revenue_musd", ascending=False)
      .round(2)
)

franchise_stats

,movie_count,total_budget_musd,mean_budget_musd,total_revenue_musd,mean_revenue_musd,mean_rating
belongs_to_collection,,,,,,
The Avengers Collection,4,1111.0,277.75,7904.08,1976.02,7.96
Star Wars Collection,2,545.0,272.50,3402.63,1701.32,6.99
Jurassic Park Collection,2,320.0,160.00,2982.01,1491.00,6.61
Avatar Collection,1,237.0,237.00,2923.71,2923.71,7.61
Frozen Collection,2,300.0,150.00,2727.90,1363.95,7.24
The Lion King (Reboot) Collection,1,260.0,260.00,1662.02,1662.02,7.09
The Fast and the Furious Collection,1,190.0,190.00,1515.40,1515.40,7.21
Black Panther Collection,1,200.0,200.00,1349.93,1349.93,7.36
Harry Potter Collection,1,125.0,125.00,1341.51,1341.51,8.10


### 3.5 Most successful directors

The tricky part here is that some movies have more than one director. The Avengers films, for example, were co-directed by Anthony and Joe Russo, so their `director` value is `Anthony Russo|Joe Russo`. If I grouped by that string directly, "Anthony Russo|Joe Russo" would count as one person.

So I split those values and gave each director their own row with `explode()`. Each co-director gets full credit for the movie's revenue. That seemed fairer than splitting the money in half, since both were fully responsible for the film. The catch is that adding up every director's revenue would count those movies twice, so I don't do that.

In [59]:
directors = (
    df.assign(director=df["director"].str.split("|"))
      .explode("director")
)

director_stats = (
    directors.groupby("director")
      .agg(
          movies_directed=("id", "nunique"),
          total_revenue_musd=("revenue_musd", "sum"),
          mean_rating=("vote_average", "mean"),
      )
      .sort_values("total_revenue_musd", ascending=False)
      .round(2)
)

director_stats.head(10)

,movies_directed,total_revenue_musd,mean_rating
director,,,
James Cameron,2,5187.87,7.76
Anthony Russo,2,4979.86,8.25
Joe Russo,2,4979.86,8.25
Joss Whedon,2,2924.22,7.68
Jennifer Lee,2,2727.90,7.24
Chris Buck,2,2727.90,7.24
J.J. Abrams,1,2068.22,7.24
Colin Trevorrow,1,1671.54,6.70
Jon Favreau,1,1662.02,7.09
